# Handlungsempfehlungen

In diesem Notebook werden die Ergebnisse der Modellierung und Risikosegmentierung in konkrete fachliche Handlungsempfehlungen für das Versicherungsportfolio übersetzt.

Ziel ist es, die Risikoprognosen nicht als automatische Entscheidungsregeln zu verwenden, sondern als Entscheidungsunterstützung für Underwriting, Portfolio-Monitoring, Pricing-Analysen und Risikopriorisierung.

Die Empfehlungen basieren auf den zuvor validierten Modell- und Segmentierungsergebnissen und müssen vor einem produktiven Einsatz zusätzlich unter fachlichen, regulatorischen und Fairness-Gesichtspunkten geprüft werden.

In [11]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [12]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RISK_SEGMENTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "output"
    / "risk_segments.pq"
)

CURATED_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "policy_curated.pq"
)

print(
    "Risk segments:",
    RISK_SEGMENTS_PATH.exists(),
)

print(
    "Curated data:",
    CURATED_PATH.exists(),
)

Risk segments: True
Curated data: True


## Operative Aktionsmatrix

Die Risikosegmente werden in eine operative Aktionsmatrix übersetzt.

Dabei dienen die Modellprognosen nicht als automatische Entscheidungsregel. Stattdessen werden unterschiedliche Prüf-, Monitoring- und Analyseintensitäten vorgeschlagen.

Die Maßnahmen werden anhand der Segmentgröße, der prognostizierten Pure Premium und des Anteils an den erwarteten Schadenkosten priorisiert.

In [13]:
risk_segments_df = pd.read_parquet(
    RISK_SEGMENTS_PATH
)

curated_df = pd.read_parquet(
    CURATED_PATH
)

recommendation_df = (
    risk_segments_df
    .merge(
        curated_df[
            [
                "IDpol",
                "BonusMalus",
                "DrivAge",
                "VehAge",
                "VehPower",
                "VehBrand",
                "VehGas",
                "Region",
                "log_density",
            ]
        ],
        on="IDpol",
        how="left",
        validate="one_to_one",
    )
)

print("Rows:", len(recommendation_df))
print(
    "Duplicate IDpol:",
    recommendation_df["IDpol"].duplicated().sum(),
)
print(
    "Missing risk segment:",
    recommendation_df["risk_segment"].isna().sum(),
)

Rows: 678013
Duplicate IDpol: 0
Missing risk segment: 0


In [14]:
action_summary = (
    recommendation_df
    .groupby(
        "risk_segment",
        observed=True,
    )
    .agg(
        policies=("IDpol", "size"),
        exposure=("Exposure", "sum"),
        mean_pure_premium=("tweedie_pp", "mean"),
        median_pure_premium=("tweedie_pp", "median"),
        expected_loss=("expected_loss_tweedie", "sum"),
        mean_bonus_malus=("BonusMalus", "mean"),
        mean_driver_age=("DrivAge", "mean"),
    )
    .reset_index()
)

action_summary["portfolio_share"] = (
    action_summary["policies"]
    / action_summary["policies"].sum()
)

action_summary["expected_loss_share"] = (
    action_summary["expected_loss"]
    / action_summary["expected_loss"].sum()
)

display(action_summary)

,risk_segment,policies,exposure,mean_pure_premium,median_pure_premium,expected_loss,mean_bonus_malus,mean_driver_age,portfolio_share,expected_loss_share
0,LOW,339008,195628.682293,84.092572,85.950912,1.632849e+07,51.851520,47.296294,0.500002,0.273824
1,STANDARD,169508,87826.421419,147.253503,144.230301,1.286717e+07,57.678723,46.776878,0.250007,0.215779
2,ELEVATED,101695,46703.607923,243.003578,235.963508,1.129960e+07,70.415674,42.221053,0.149990,0.189491
3,HIGH,67802,28201.393828,711.282388,479.381063,1.913611e+07,88.538214,38.235583,0.100001,0.320907


In [15]:
action_matrix = pd.DataFrame({
    "risk_segment": [
        "LOW",
        "STANDARD",
        "ELEVATED",
        "HIGH",
    ],
    "priority": [
        "Standard",
        "Moderate",
        "High",
        "Very High",
    ],
    "recommended_action": [
        "Standard process",
        "Standard process",
        "Additional review recommended",
        "Manual underwriting review recommended",
    ],
    "monitoring_focus": [
        "Portfolio stability",
        "Migration into higher risk segments",
        "Risk drivers and pricing adequacy",
        "Risk concentration, pricing adequacy and segment development",
    ],
    "decision_rule": [
        "Decision support only",
        "Decision support only",
        "Model-supported additional review",
        "Manual review supported by model",
    ],
})

display(action_matrix)

,risk_segment,priority,recommended_action,monitoring_focus,decision_rule
0,LOW,Standard,Standard process,Portfolio stability,Decision support only
1,STANDARD,Moderate,Standard process,Migration into higher risk segments,Decision support only
2,ELEVATED,High,Additional review recommended,Risk drivers and pricing adequacy,Model-supported additional review
3,HIGH,Very High,Manual underwriting review recommended,"Risk concentration, pricing adequacy and segme...",Manual review supported by model


### Interpretation der operativen Prioritäten

Die Segmentstruktur ermöglicht eine abgestufte operative Bearbeitung des Portfolios.

Das `LOW`-Segment umfasst rund die Hälfte aller Policen und trägt etwa 27,4 % der prognostizierten Schadenkosten. Für dieses Segment steht daher ein effizienter Standardprozess im Vordergrund.

Das `STANDARD`-Segment umfasst rund 25 % des Portfolios und rund 21,6 % der erwarteten Schadenkosten. Auch hier ist grundsätzlich ein Standardprozess ausreichend, ergänzt durch reguläres Monitoring und segmentbasierte Vergleichsanalysen.

Das `ELEVATED`-Segment enthält rund 15 % der Policen und trägt etwa 18,9 % der erwarteten Schadenkosten. Für diese Policen wird eine zusätzliche risikoorientierte Prüfung empfohlen.

Besonders relevant ist das `HIGH`-Segment: Rund 10 % aller Policen tragen etwa 32,1 % der prognostizierten Schadenkosten. Für dieses Segment wird daher eine manuelle Underwriting-Prüfung empfohlen, ergänzt durch intensiveres Portfolio-Monitoring und Pricing-Analysen.

Die Risikosegmente dienen ausschließlich als Entscheidungsunterstützung und nicht als automatische Grundlage für Annahme-, Ablehnungs- oder Preisentscheidungen.

## Messbare Steuerungs-KPIs

Die Handlungsempfehlungen werden mit messbaren Portfolio-KPIs verknüpft.

Neben Segmentgröße, durchschnittlicher Pure Premium und erwartetem Schadenanteil wird ein `Loss Concentration Index` verwendet. Dieser setzt den Anteil der erwarteten Schadenkosten ins Verhältnis zum Anteil der Policen.

Ein Wert über 1 bedeutet, dass ein Segment relativ zu seiner Größe überproportional viele erwartete Schadenkosten trägt.

In [16]:
management_kpi = action_summary.copy()

management_kpi["expected_loss_per_policy"] = (
    management_kpi["expected_loss"]
    / management_kpi["policies"]
)

management_kpi["expected_loss_per_exposure"] = (
    management_kpi["expected_loss"]
    / management_kpi["exposure"]
)

management_kpi["loss_concentration_index"] = (
    management_kpi["expected_loss_share"]
    / management_kpi["portfolio_share"]
)

management_kpi = management_kpi.merge(
    action_matrix[
        [
            "risk_segment",
            "priority",
            "recommended_action",
            "monitoring_focus",
            "decision_rule",
        ]
    ],
    on="risk_segment",
    how="left",
    validate="one_to_one",
)

management_kpi = management_kpi[
    [
        "risk_segment",
        "priority",
        "policies",
        "portfolio_share",
        "exposure",
        "mean_pure_premium",
        "expected_loss",
        "expected_loss_share",
        "expected_loss_per_policy",
        "expected_loss_per_exposure",
        "loss_concentration_index",
        "recommended_action",
        "monitoring_focus",
        "decision_rule",
    ]
]

display(management_kpi)

,risk_segment,priority,policies,portfolio_share,exposure,mean_pure_premium,expected_loss,expected_loss_share,expected_loss_per_policy,expected_loss_per_exposure,loss_concentration_index,recommended_action,monitoring_focus,decision_rule
0,LOW,Standard,339008,0.500002,195628.682293,84.092572,1.632849e+07,0.273824,48.165487,83.466724,0.547645,Standard process,Portfolio stability,Decision support only
1,STANDARD,Moderate,169508,0.250007,87826.421419,147.253503,1.286717e+07,0.215779,75.908917,146.506809,0.863090,Standard process,Migration into higher risk segments,Decision support only
2,ELEVATED,High,101695,0.149990,46703.607923,243.003578,1.129960e+07,0.189491,111.112623,241.942725,1.263359,Additional review recommended,Risk drivers and pricing adequacy,Model-supported additional review
3,HIGH,Very High,67802,0.100001,28201.393828,711.282388,1.913611e+07,0.320907,282.235144,678.551824,3.209035,Manual underwriting review recommended,"Risk concentration, pricing adequacy and segme...",Manual review supported by model


### Ergebnis der Steuerungs-KPIs

Die Steuerungs-KPIs bestätigen die starke Risikokonzentration in den oberen Segmenten.

Der `Loss Concentration Index` steigt von rund 0,55 im `LOW`-Segment über rund 0,86 im `STANDARD`-Segment auf rund 1,26 im `ELEVATED`-Segment und erreicht im `HIGH`-Segment rund 3,21. Damit trägt das höchste Risikosegment relativ zu seinem Anteil an den Policen mehr als dreimal so viele erwartete Schadenkosten.

Auch die exposure-bezogenen erwarteten Schadenkosten unterscheiden sich deutlich. Sie steigen von rund 83 Euro im `LOW`-Segment auf rund 679 Euro im `HIGH`-Segment.

Damit können die Risikosegmente nicht nur zur Beschreibung des Portfolios, sondern auch zur priorisierten Steuerung von Underwriting-Reviews, Pricing-Analysen und Portfolio-Monitoring verwendet werden.

Die Kennzahlen basieren auf prognostizierten erwarteten Schadenkosten und stellen keine beobachteten zukünftigen Schadenresultate dar.

In [17]:
MANAGEMENT_KPI_PATH = (
    PROJECT_ROOT
    / "data"
    / "output"
    / "management_kpi.pq"
)

POLICY_ACTION_PATH = (
    PROJECT_ROOT
    / "data"
    / "output"
    / "policy_action_priorities.pq"
)


# Segment-level management table
management_kpi.to_parquet(
    MANAGEMENT_KPI_PATH,
    index=False,
)


# Policy-level operational priority table
policy_action_priorities = (
    risk_segments_df
    .merge(
        action_matrix[
            [
                "risk_segment",
                "priority",
                "recommended_action",
                "monitoring_focus",
                "decision_rule",
            ]
        ],
        on="risk_segment",
        how="left",
        validate="many_to_one",
    )
)

policy_action_priorities.to_parquet(
    POLICY_ACTION_PATH,
    index=False,
)


print(
    "Management KPI shape:",
    management_kpi.shape,
)

print(
    "Policy actions shape:",
    policy_action_priorities.shape,
)

print(
    "Duplicate policy IDs:",
    policy_action_priorities["IDpol"]
    .duplicated()
    .sum(),
)

print(
    "Missing priorities:",
    policy_action_priorities["priority"]
    .isna()
    .sum(),
)

print(
    "Management KPI saved:",
    MANAGEMENT_KPI_PATH.exists(),
)

print(
    "Policy actions saved:",
    POLICY_ACTION_PATH.exists(),
)

print(
    "\nManagement KPI:",
    MANAGEMENT_KPI_PATH.relative_to(PROJECT_ROOT),
)

print(
    "Policy actions:",
    POLICY_ACTION_PATH.relative_to(PROJECT_ROOT),
)

Management KPI shape: (4, 14)
Policy actions shape: (678013, 11)
Duplicate policy IDs: 0
Missing priorities: 0
Management KPI saved: True
Policy actions saved: True

Management KPI: data\output\management_kpi.pq
Policy actions: data\output\policy_action_priorities.pq


In [18]:
print("Management KPI shape:", management_kpi.shape)
print("Policy actions shape:", policy_action_priorities.shape)
print(
    "Missing priorities:",
    policy_action_priorities["priority"].isna().sum(),
)
print(
    "\nRisk segments:"
)
print(
    policy_action_priorities["risk_segment"]
    .value_counts(sort=False)
)

Management KPI shape: (4, 14)
Policy actions shape: (678013, 11)
Missing priorities: 0

Risk segments:
risk_segment
LOW         339008
STANDARD    169508
HIGH         67802
ELEVATED    101695
Name: count, dtype: int64


In [19]:
display(
    policy_action_priorities[
        [
            "risk_segment",
            "priority",
            "recommended_action",
            "decision_rule",
        ]
    ]
    .drop_duplicates()
    .sort_values("risk_segment")
)

,risk_segment,priority,recommended_action,decision_rule
24,ELEVATED,High,Additional review recommended,Model-supported additional review
14,HIGH,Very High,Manual underwriting review recommended,Manual review supported by model
0,LOW,Standard,Standard process,Decision support only
2,STANDARD,Moderate,Standard process,Decision support only


## Fazit – Handlungsempfehlungen

Die Modell- und Segmentierungsergebnisse wurden in eine operative, risikoorientierte Steuerungslogik für das Versicherungsportfolio übersetzt.

Die vier Risikosegmente `LOW`, `STANDARD`, `ELEVATED` und `HIGH` unterscheiden sich deutlich hinsichtlich prognostizierter Pure Premium, erwarteter Schadenkosten und Risikokonzentration. Besonders das `HIGH`-Segment besitzt eine hohe operative Relevanz: Es umfasst rund 10 % aller Policen, trägt jedoch rund 32,1 % der prognostizierten erwarteten Schadenkosten. Der `Loss Concentration Index` beträgt rund 3,21.

Daraus ergibt sich eine abgestufte Bearbeitungslogik. Für `LOW`- und `STANDARD`-Policen ist grundsätzlich ein Standardprozess vorgesehen. Für das `ELEVATED`-Segment wird eine zusätzliche risikoorientierte Prüfung empfohlen. Für das `HIGH`-Segment wird eine manuelle Underwriting-Prüfung empfohlen, ergänzt durch intensiveres Portfolio-Monitoring und Pricing-Analysen.

Die Modellprognosen stellen dabei keine automatischen Annahme-, Ablehnungs- oder Preisentscheidungen dar. Sie dienen ausschließlich als analytische Entscheidungsunterstützung und müssen im produktiven Einsatz durch fachliche Prüfung sowie regulatorische und Fairness-Kontrollen ergänzt werden.

Für die weitere technische Nutzung wurden sowohl eine aggregierte Management-KPI-Tabelle als auch eine Policy-level-Tabelle mit operativen Prioritäten als reproduzierbare Datenartefakte gespeichert:

- `data/output/management_kpi.pq`
- `data/output/policy_action_priorities.pq`

Die Policy-level-Tabelle enthält zusätzlich die Risikodezile und ermöglicht damit neben der breiteren Segmentlogik auch eine feinere Priorisierung innerhalb des Portfolios.

Damit verbindet die Plattform die statistische Risikomodellierung mit konkreter Portfolio-Steuerung und schafft eine Grundlage für die nachfolgenden API- und Power-BI-Komponenten.